# NimbusPay Ticket Triage — free Colab T4
Run this notebook on a Colab T4 GPU. See `COLAB_INSTRUCTIONS.md` for the full rationale and submission constraints.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload nimbuspay_colab_project.zip


Saving nimbuspay_colab_project_v2.zip to nimbuspay_colab_project_v2.zip


In [ ]:
!unzip -q nimbuspay_colab_project_v2.zip -d /content
%cd /content/nimbuspay_colab_project
!nvidia-smi
!pip -q install -r requirements.txt


/content/nimbuspay_colab_project
Thu Nov 12 08:15:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------

In [ ]:
# Audit, reconstruct the 25 truncated labels, remove train/dev input leakage, and log every change.
!python clean_data.py
!head -12 artifacts/cleaning_log.csv
!python prepare_raw.py


Wrote 1355 cleaned rows and 370 log entries.
Dropped 90 rows; fixed 280 rows.
id,action,reason
tr-00008,drop,exact train/dev input overlap; dev labels were not read
tr-00013,fix,recomputed priority from explicit SCHEMA.md rules
tr-00018,fix,recomputed priority from explicit SCHEMA.md rules
tr-00020,fix,renamed invalid key prio to priority
tr-00020,fix,normalized txn_date to required YYYY-MM-DD
tr-00020,fix,recomputed priority from explicit SCHEMA.md rules
tr-00023,fix,recomputed priority from explicit SCHEMA.md rules
tr-00045,fix,recomputed priority from explicit SCHEMA.md rules
tr-00049,fix,renamed invalid key prio to priority
tr-00049,fix,recomputed priority from explicit SCHEMA.md rules
tr-00058,fix,renamed invalid key prio to priority
Wrote 1420 parseable but otherwise raw rows to artifacts/train_raw_parseable.jsonl.


In [ ]:
# Required baselines. Rules are allowed only for the second baseline.
!python baseline.py --output artifacts/dev_base_fixed.jsonl
!python score.py --gold data/dev.jsonl --pred artifacts/dev_base_fixed.jsonl
!python baseline.py --with-rules --output artifacts/dev_base_rules.jsonl
!python score.py --gold data/dev.jsonl --pred artifacts/dev_base_rules.jsonl


config.json: 100% 661/661 [00:00<00:00, 3.59MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 25.1MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 67.2MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 107MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 136MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 86.6MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  40% 2.50G/6.17G [00:30<00:47, 77.2MB/s, 42.3MB/s  ]

Fetching 2 files:  50% 1/2 [00:46<00:46, 46.02s/it]
Reconstructing (incomplete total...):  95% 5.84G/6.17G [00:53<00:02, 156MB/s,  124MB/s  ]
Reconstructing (incomplete total...): 100% 6.17G/6.17G [00:54<00:00, 171MB/s,  156MB/s  ]

Fetching 2 files: 100% 2/2 [00:55<00:00, 27.67s/it]
Download 

In [ ]:
from pathlib import Path

path = Path("/content/nimbuspay_colab_project/train.py")
text = path.read_text()

old = '''        prompt_ids = tokenizer.apply_chat_template(messages[:2], tokenize=True, add_generation_prompt=True)
'''
new = '''        prompt_encoded = tokenizer.apply_chat_template(messages[:2], tokenize=True, add_generation_prompt=True)
        prompt_ids = prompt_encoded["input_ids"] if isinstance(prompt_encoded, dict) else prompt_encoded
        if prompt_ids and isinstance(prompt_ids[0], list):
            prompt_ids = prompt_ids[0]
'''

assert old in text, "Expected training-code line was not found."
path.write_text(text.replace(old, new))

print("Patched train.py successfully.")

Patched train.py successfully.


In [ ]:
from pathlib import Path

path = Path("/content/nimbuspay_colab_project/train.py")
text = path.read_text()

start = text.index("        prompt_encoded = tokenizer.apply_chat_template")
end = text.index("        target_ids =", start)

replacement = '''        prompt_encoded = tokenizer.apply_chat_template(
            messages[:2], tokenize=True, add_generation_prompt=True
        )
        try:
            prompt_ids = prompt_encoded["input_ids"]  # BatchEncoding on newer Transformers
        except (TypeError, KeyError):
            prompt_ids = prompt_encoded              # list on older Transformers
        if prompt_ids and isinstance(prompt_ids[0], list):
            prompt_ids = prompt_ids[0]
'''

text = text[:start] + replacement + text[end:]
path.write_text(text)

print(path.read_text()[start:start + 600])

        prompt_encoded = tokenizer.apply_chat_template(
            messages[:2], tokenize=True, add_generation_prompt=True
        )
        try:
            prompt_ids = prompt_encoded["input_ids"]  # BatchEncoding on newer Transformers
        except (TypeError, KeyError):
            prompt_ids = prompt_encoded              # list on older Transformers
        if prompt_ids and isinstance(prompt_ids[0], list):
            prompt_ids = prompt_ids[0]
        target_ids = tokenizer(messages[2]["content"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
        # Preserve the syst


In [ ]:
from pathlib import Path

path = Path("/content/nimbuspay_colab_project/train.py")
text = path.read_text()

assert 'warmup_ratio=0.05' in text
path.write_text(text.replace('warmup_ratio=0.05', 'warmup_steps=9'))

print("Patched warmup setting.")

Patched warmup setting.


In [ ]:
from pathlib import Path

path = Path("/content/nimbuspay_colab_project/train.py")
text = path.read_text()

text = text.replace("import argparse\n", "import argparse\nimport inspect\n")

start = text.index("    train_args = TrainingArguments(")
end = text.index("    trainer = Trainer(", start)

replacement = '''    train_kwargs = dict(
        output_dir=args.output + "_checkpoints",
        num_train_epochs=args.epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=args.learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=9,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
        fp16=True,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        group_by_length=True,
        save_total_limit=1,
        seed=args.seed,
    )
    supported = set(inspect.signature(TrainingArguments).parameters)
    skipped = sorted(set(train_kwargs) - supported)
    if skipped:
        print("Skipping unsupported optional settings:", skipped)
    train_args = TrainingArguments(
        **{key: value for key, value in train_kwargs.items() if key in supported}
    )
'''

path.write_text(text[:start] + replacement + text[end:])
print("Patched TrainingArguments compatibility.")

Patched TrainingArguments compatibility.


In [ ]:
# # Final QLoRA adapter: 3B base model, rank 8, 4-bit NF4, assistant-only loss.
# !python train.py --train artifacts/train_clean.jsonl --output artifacts/adapter_final --load-in-4bit --rank 8 --epochs 2 --learning-rate 2e-4 --max-length 2048

!python train.py --train artifacts/train_clean.jsonl --output artifacts/adapter_final --load-in-4bit --rank 8 --epochs 2 --learning-rate 2e-4 --max-length 2048

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:08<00:00, 51.70it/s]
trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827
Tokenized 1355 examples; truncated long user prompts: 7
Skipping unsupported optional settings: ['group_by_length']
{'loss': '2.051', 'grad_norm': '2.205', 'learning_rate': '0.0002', 'epoch': '0.1181'}
{'loss': '0.2479', 'grad_norm': '0.4364', 'learning_rate': '0.0001981', 'epoch': '0.2362'}
{'loss': '0.09163', 'grad_norm': '0.5854', 'learning_rate': '0.0001925', 'epoch': '0.3542'}
{'loss': '0.0725', 'grad_norm': '0.3663', 'learning_rate': '0.0001833', 'epoch': '0.4723'}
{'loss': '0.05314', 'grad_norm': '0.546', 'learning_rate': '0.0001711', 'epoch': '0.5904'}
{'loss': '0.04665', 'grad_norm': '0.2836', 'learning_rate': '0.0001561', 'epoch': '0.7085'}
{'loss': '0.03118', 'grad_norm': '0.3291', 'learning_rate': '0.0001389', 'epoch': '0.8266'}
{'loss': '0.02618', 'grad_norm': '0.1419', 'learn

In [ ]:
# Development evaluation
!python predict.py --input data/dev.jsonl --adapter artifacts/adapter_final --output artifacts/dev_final_predictions.jsonl
!python score.py --gold data/dev.jsonl --pred artifacts/dev_final_predictions.jsonl


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:08<00:00, 49.59it/s]
group                            n   valid   exact  fields
all                            200   85.5%   83.5%   85.2%
category=account_access         19   57.9%   52.6%   57.2%
category=fraud                  15  100.0%  100.0%  100.0%
category=kyc                    24   70.8%   70.8%   70.8%
category=offers                 18   94.4%   94.4%   94.4%
category=other                  16   68.8%   68.8%   68.8%
category=payment_failed         53   92.5%   90.6%   92.2%
category=refund                 55   92.7%   89.1%   92.3%

per-field accuracy (all rows)
  category        85.5%
  priority        85.5%
  amount_paise    84.5%
  txn_date        85.5%
  txn_id          85.5%
  channel         85.0%
  language        85.0%
  needs_human     85.5%


In [ ]:
# Hidden-test submission predictions
!python predict.py --input data/test_inputs.jsonl --adapter artifacts/adapter_final --output artifacts/predictions_test.jsonl


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:08<00:00, 49.47it/s]


In [ ]:
%cd /content/nimbuspay_colab_project

!zip -rq /content/nimbuspay_results.zip artifacts

from google.colab import files
files.download("/content/nimbuspay_results.zip")

/content/nimbuspay_colab_project


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -rq /content/nimbuspay_submission.zip \
  artifacts/adapter_final \
  artifacts/predictions_test.jsonl \
  artifacts/cleaning_log.csv

from google.colab import files
files.download("/content/nimbuspay_submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Controlled experiments
Use the commands in `COLAB_INSTRUCTIONS.md` for raw-vs-cleaned, 16-bit-vs-4-bit, rank, and epoch experiments. Change `--output` per run and record the scorer metrics, peak GPU memory, and wall-clock time.